# DM-AHR-ViT — full pipeline on Modal

This notebook reproduces the study end to end. Every GPU job runs on Modal inside the app
`modal_app.py`; nothing here needs a local GPU, and the notebook itself can be closed while jobs
run because they are launched with `--detach`.

**The runtime you pick here does not matter.** Every cell below shells out to `modal run`, so the
notebook has no local compute to accelerate: selecting an A100 in Colab accelerates nothing and
still bills Modal for the actual work. A free CPU runtime is enough. What the session must do is
stay connected long enough for the non-detached calls (evaluation, statistics, report) to return.

**Cost warning.** A complete rerun is roughly 8–14 A100-hours, dominated by Arm C diffusion
sampling over 8,993 hair-present images. Read `reports/compute_plan.md` before starting, and check
the account balance first. Do not launch phase 6 unless the earlier phases produced their
artifacts.

**Research prototype only. Not clinically validated, not for diagnosis or patient care.**

## 1. Setup

You need: a Modal account with a token, the repository, and nothing else. The token is read from
the environment; this notebook never prints it.

**Credential check.** Reports whether a Modal token is present in the environment,
without ever printing its value.

In [ ]:
import os
from pathlib import Path

# --- fill in only if you are on a fresh runtime without the variable set ---
# os.environ["MODAL_TOKEN_ID"] = "..."
# os.environ["MODAL_TOKEN_SECRET"] = "..."

print("MODAL_API_TOKEN set:", bool(os.environ.get("MODAL_API_TOKEN")))
print("MODAL_TOKEN_ID set:  ", bool(os.environ.get("MODAL_TOKEN_ID")))
print("MODAL_TOKEN_SECRET set:", bool(os.environ.get("MODAL_TOKEN_SECRET")))

**Install the Modal client.** Pins nothing, so check the printed version — the app uses
the v1 API names.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modal"], check=True)
print(subprocess.run(["modal", "--version"], capture_output=True, text=True).stdout.strip())

**Get the code.** Clones the pipeline branch and installs its pinned requirements.

In [ ]:
# Clone the project if it is not already here, then enter it.
REPO_URL = "https://github.com/supraehhe-web/ML-shit.git"
BRANCH = "hoplite/dm-ahr-vit-pipeline"

if not Path("/content/ML-shit").exists():
    !git clone --branch {BRANCH} {REPO_URL} /content/ML-shit
%cd /content/ML-shit
!pip install -q -r requirements.txt

**Confirm access.** Shows the active Modal profile and lists the volume, so a credential
or naming problem surfaces here rather than mid-run.

In [ ]:
!modal profile current          # confirms the active token
!modal volume ls dmahr-vit-vol  # confirms the volume and its contents

## 2. Verify the GPU from inside a Modal function

`AGENTS.md` requires the GPU to be established by running code on Modal, not by trusting a
config string. The result is written to `artifacts/logs/environment_report.json`.

**Verify the GPU from inside Modal.** The hardware is established by running code on
Modal, not by trusting the config string, and the result is written to
`artifacts/logs/environment_report.json`.

In [ ]:
!modal run modal_app.py::gpu_probe

## 3. Data acquisition

Downloads ISIC collection 212 (11,720 images) and the HAM10000 masks into the volume. Raw data is
immutable and never enters Git.

`--limit` is available for a smoke run; a limited run writes a different manifest and must not be
mixed with full results.

**Phase 1 — download.** Fetches collection 212 and the HAM10000 metadata and masks into
the volume. Raw data is immutable and never enters Git.

In [ ]:
!modal run modal_app.py::download_data

## 4. Audit, manifests and the frozen split

Recomputes every count from the files, merges confirmed near-duplicates into lesion groups, builds
the class-stratified 70/15/15 split, and proves zero group overlap. The split hash is frozen so any
later change is detectable.

**Phase 2 — audit, manifests and frozen split.** Recomputes every count from the files,
merges confirmed duplicates, and proves zero group overlap before hashing the split.

In [ ]:
!modal run modal_app.py::build_manifests

## 5. Hair screening, Arm B and the synthetic pairs

Black-hat/top-hat screening decides hair presence, then DullRazor-style morphological detection plus
Telea inpainting produces Arm B. Synthetic hairy/hairless pairs are composited only from hair-absent
images, with a recorded seed.

The screening threshold is frozen here, before any test metric exists.

**Phase 3 — screening, Arm B and synthetic pairs.** The detector threshold is frozen
here, before any test metric exists.

In [ ]:
!modal run modal_app.py::hair_pipeline

## 6. DM-AHR audit and smoke test

Verifies the pinned upstream commit, parses the upstream config, loads the released checkpoint, and
confirms a batch round-trip is finite and reproducible. Two upstream bugs are worked around here and
documented in the decision log: a sampler that returns a single image per batch, and a diffusion
schedule that is never initialised.

Never skip this on a new machine — phase 8 assumes the workaround.

**Phase 4 — audit and smoke test.** Confirms the pinned upstream commit, parses its
config, loads the released checkpoint and checks a batch is finite and reproducible.

In [ ]:
!modal run modal_app.py::dmahr_smoke
!modal run modal_app.py::dmahr_bench

## 7. Train DM-AHR on train/validation sources only

Leakage control: the model that produces Arm C is trained here, on train-split sources with
validation for model selection. The released weights are used for the smoke test only, because their
training data is undocumented.

**Phase 5 — diffusion training.** Leakage-controlled: train-split sources only, model
selection on validation. Detached so it survives a dropped connection.

In [ ]:
# budget-forced ablation: upstream config uses 660,000 iterations; see decision log D1/D12
!modal run --detach modal_app.py::dmahr_train

## 8. Generate Arm C

Samples 8,993 hair-present images plus the held-out test pairs. Sharded across containers; each shard
skips images already written, so a preempted run resumes where it stopped.

`--num-timesteps 1000` is the budget-forced ablation used for the reported results. Every per-image
row records its `num_timesteps` so the deviation is auditable. Drop the flag for the 2,000-step
reference settings at roughly double the cost.

**Phase 6 — Arm C generation.** Sharded across containers, each shard resuming from what
is already written. This is the expensive phase.

In [ ]:
!modal run --detach modal_app.py::dmahr_sample_driver_ablation

## 9. Fill Arm C for hair-absent images

Those images are identical in every arm, so they are copied through rather than sampled. Without this
step the Arm C dataset is missing images and ViT training fails.

**Phase 6 — passthrough.** Copies hair-absent images into Arm C, where they are
identical to the originals.

In [ ]:
!modal run modal_app.py::fill_c_passthrough

## 10. Train the ViT classifiers

Four arms × three seeds, identical recipe. Each run is resumable and writes its own checkpoint
directory with the config, seed, class weights and git commit.

**Phase 7 — ViT training.** Four arms by three seeds, identical recipe, each detached and
resumable.

In [ ]:
for arm in ["A", "Astar", "B", "C"]:
    for seed in [42, 123, 2026]:
        print(f"launching arm {arm} seed {seed}")
        !modal run --detach modal_app.py::vit_train --arm {arm} --seed {seed}

## 11. Evaluate on the locked test set

Per-image predictions and metrics. The test split is scored once per frozen checkpoint.

**Phase 8 — locked-test evaluation.** Per-image predictions and metrics, scored once per
frozen checkpoint.

In [ ]:
for arm in ["A", "Astar", "B", "C"]:
    for seed in [42, 123, 2026]:
        !modal run modal_app.py::vit_eval --arm {arm} --seed {seed}

## 12. Paired image quality and explanations

PSNR/SSIM against the synthetic hairless references, then Grad-CAM and attention roll-out with
lesion-overlap, pointing game and chance baselines.

**Phase 9 — paired image quality.** PSNR and SSIM against the synthetic hairless
references.

In [ ]:
!modal run modal_app.py::image_quality --split test

**Phase 9 — explainability.** Grad-CAM and attention roll-out with lesion-overlap,
pointing game and chance baselines.

In [ ]:
for arm in ["A", "Astar", "B", "C"]:
    for seed in [42, 123, 2026]:
        !modal run --detach modal_app.py::xai_run --arm {arm} --seed {seed}

## 13. Statistics, report and gates

Bootstrap plus Wilcoxon, both Holm-corrected; then the report, the artifact inventory with
SHA-256 hashes, and the acceptance check.

**Phase 10 — statistics, report and gates.** Bootstrap and Wilcoxon with Holm
correction, then the report, the hashed inventory and the acceptance check.

In [ ]:
!modal run modal_app.py::stats_run
!modal run modal_app.py::finalize
!modal run modal_app.py::acceptance

## 14. Retrieve the results

Checkpoints and processed images stay on the volume by design. Pull the small artifacts into the
notebook runtime; `01_results_walkthrough.ipynb` can then explain them without any GPU.

**Retrieve results.** Pulls the small artifacts from the volume into the notebook
runtime; checkpoints and processed images stay on the volume by design.

In [ ]:
!mkdir -p artifacts/logs artifacts/predictions artifacts/panels artifacts/reports
!modal volume get dmahr-vit-vol /artifacts/logs artifacts/logs --force
!modal volume get dmahr-vit-vol /artifacts/predictions artifacts/predictions --force
!modal volume get dmahr-vit-vol /artifacts/panels artifacts/panels --force
!modal volume get dmahr-vit-vol /artifacts/reports artifacts/reports --force

**Retrieve the split manifests.** They are git-ignored, so they are fetched separately
when a split-level analysis is needed.

In [ ]:
# Split manifests are git-ignored; fetch them separately for split-level analysis.
!modal volume get dmahr-vit-vol /data/manifests data/manifests --force

## 15. Before you re-run anything

- Check the account balance; a full Arm C rerun is the expensive step.
- Do not launch two drivers for the same phase. Shards skip finished images, but duplicate
  containers still bill GPU time.
- A phase that produced no artifact must not be followed by the next phase. The acceptance checker
  exists to catch exactly this.